In [1]:
import math
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use NLTK for tokenization and stopword removal.
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    # Fallback keeps the notebook usable if the NLTK stopwords corpus
    # has not yet been downloaded in the execution environment.
    STOP_WORDS = {
        "a","an","and","are","as","at","be","been","by","for","from",
        "has","have","he","her","his","i","if","in","is","it","its",
        "me","my","of","on","or","our","she","that","the","their",
        "them","there","these","they","this","to","was","we","were",
        "what","when","where","which","who","will","with","you","your"
    }

def preprocess(text):
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    return [t for t in tokens if t not in STOP_WORDS]

def clean_text(text):
    return " ".join(preprocess(text))

print("Libraries loaded successfully.")
print("Stopwords available:", len(STOP_WORDS))


Libraries loaded successfully.
Stopwords available: 198


In [2]:
# Q3(a): 40-sentence corpus.
sentences = [
    "Machine learning models learn patterns from data.",
    "Machine learning algorithms improve prediction accuracy.",
    "Deep learning models use neural networks.",
    "Neural networks learn complex patterns from data.",
    "Artificial intelligence systems solve complex problems.",
    "Artificial intelligence uses machine learning methods.",
    "Data science combines statistics and programming.",
    "Data analysis helps researchers find useful patterns.",
    "Data scientists build models from large datasets.",
    "Researchers analyze data to test new ideas.",
    "Computer vision systems analyze images.",
    "Computer vision models detect objects in images.",
    "Image classification assigns labels to images.",
    "Deep learning improves image classification performance.",
    "Natural language processing analyzes human language.",
    "Language models learn patterns from text.",
    "Natural language systems process words and sentences.",
    "Text classification assigns categories to documents.",
    "Students learn programming in computer science courses.",
    "Students practice algorithms and data structures.",
    "Programming skills help students solve problems.",
    "Computer science students study machine learning.",
    "Online courses provide lectures and programming exercises.",
    "Online education helps students learn new skills.",
    "Education platforms use data to track student progress.",
    "Teachers use data analysis to understand learning outcomes.",
    "Algorithms process data to produce useful results.",
    "Algorithms can classify images and text.",
    "Models learn from training data.",
    "Training data improves machine learning models.",
    "Large datasets help neural networks learn.",
    "Researchers train models on large datasets.",
    "Prediction models estimate future outcomes.",
    "Machine learning supports prediction in many applications.",
    "Artificial intelligence research develops new models.",
    "Researchers develop algorithms for language processing.",
    "Language models represent words using their contexts.",
    "Words with similar contexts can have related meanings.",
    "Context helps a model represent the meaning of words.",
    "Distributional representations describe words through context."
]

processed = [preprocess(s) for s in sentences]
print("Number of sentences:", len(sentences))
print("First five processed sentences:")
for s in processed[:5]:
    print(s)


Number of sentences: 40
First five processed sentences:
['machine', 'learning', 'models', 'learn', 'patterns', 'data']
['machine', 'learning', 'algorithms', 'improve', 'prediction', 'accuracy']
['deep', 'learning', 'models', 'use', 'neural', 'networks']
['neural', 'networks', 'learn', 'complex', 'patterns', 'data']
['artificial', 'intelligence', 'systems', 'solve', 'complex', 'problems']


In [3]:
# Q3(b/d): Build vocabulary and identify frequent words.
vocabulary = sorted(set(word for sent in processed for word in sent))
counts = pd.Series([word for sent in processed for word in sent]).value_counts()

print("Vocabulary size:", len(vocabulary))
print("\nTop 15 frequent words:")
display(counts.head(15).to_frame("Frequency"))

selected_words = counts.head(15).index.tolist()
print("Selected words:", selected_words)


Vocabulary size: 105

Top 15 frequent words:


,Frequency
data,12
models,11
learning,9
learn,7
machine,6
language,6
algorithms,5
words,5
students,5
patterns,4


Selected words: ['data', 'models', 'learning', 'learn', 'machine', 'language', 'algorithms', 'words', 'students', 'patterns', 'programming', 'researchers', 'computer', 'images', 'prediction']


In [4]:
# Q3(c/e): Build a word-context co-occurrence matrix.
# Context window = two words before and two words after the target word.

window_size = 2

cooc = pd.DataFrame(
    0.0,
    index=vocabulary,
    columns=vocabulary
)

for tokens in processed:
    for i, target in enumerate(tokens):
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)

        for j in range(start, end):
            if i != j:
                context = tokens[j]
                cooc.loc[target, context] += 1

print("Co-occurrence matrix dimensions:", cooc.shape)
print("\nVectors of selected frequent words:")
display(cooc.loc[selected_words].round(2))


Co-occurrence matrix dimensions: (105, 105)

Vectors of selected frequent words:


,accuracy,algorithms,analysis,analyze,analyzes,applications,artificial,assigns,build,categories,...,track,train,training,understand,use,useful,uses,using,vision,words
data,0.0,2.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,2.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0
models,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
learning,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
learn,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
machine,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
language,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
algorithms,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
words,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
students,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
patterns,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [5]:
# Q3(f): Cosine similarity between selected word vectors.
word_vectors = cooc.loc[selected_words].values
word_similarity = cosine_similarity(word_vectors)

word_similarity_df = pd.DataFrame(
    word_similarity,
    index=selected_words,
    columns=selected_words
)

display(word_similarity_df.round(4))


,data,models,learning,learn,machine,language,algorithms,words,students,patterns,programming,researchers,computer,images,prediction
data,1.0000,0.3906,0.3085,0.2445,0.0632,0.1537,0.1738,0.0348,0.3823,0.2574,0.1778,0.3264,0.1576,0.1474,0.0983
models,0.3906,1.0000,0.3098,0.3195,0.3406,0.1105,0.2810,0.0624,0.3122,0.2601,0.1597,0.0419,0.1770,0.0883,0.1766
learning,0.3085,0.3098,1.0000,0.1670,0.2286,0.2053,0.2321,0.0870,0.2901,0.2417,0.0371,0.1946,0.1316,0.0410,0.2462
learn,0.2445,0.3195,0.1670,1.0000,0.3446,0.1886,0.2843,0.1066,0.1777,0.3948,0.1818,0.2860,0.2417,0.0503,0.2010
machine,0.0632,0.3406,0.2286,0.3446,1.0000,0.1489,0.3030,0.0673,0.0673,0.1870,0.0431,0.1807,0.1145,0.0476,0.4762
language,0.1537,0.1105,0.2053,0.1886,0.1489,1.0000,0.1966,0.2949,0.1474,0.3185,0.0629,0.2638,0.2229,0.1390,0.2085
algorithms,0.1738,0.2810,0.2321,0.2843,0.3030,0.1966,1.0000,0.0556,0.1111,0.2057,0.0711,0.2236,0.0630,0.0786,0.1571
words,0.0348,0.0624,0.0870,0.1066,0.0673,0.2949,0.0556,1.0000,0.0000,0.1029,0.0000,0.0745,0.1260,0.0786,0.0786
students,0.3823,0.3122,0.2901,0.1777,0.0673,0.1474,0.1111,0.0000,1.0000,0.3086,0.4264,0.1491,0.3150,0.0786,0.0786
patterns,0.2574,0.2601,0.2417,0.3948,0.1870,0.3185,0.2057,0.1029,0.3086,1.0000,0.1974,0.4140,0.2916,0.0727,0.1455


In [6]:
# Q3(g): Identify the most similar word pairs.
pairs = []

for i in range(len(selected_words)):
    for j in range(i + 1, len(selected_words)):
        pairs.append([
            selected_words[i],
            selected_words[j],
            word_similarity[i, j]
        ])

pair_table = pd.DataFrame(
    pairs,
    columns=["Word 1", "Word 2", "Cosine Similarity"]
).sort_values("Cosine Similarity", ascending=False)

display(pair_table.head(10).round(4))


,Word 1,Word 2,Cosine Similarity
59,machine,prediction,0.4762
85,students,programming,0.4264
91,patterns,researchers,0.4140
44,learn,patterns,0.3948
0,data,models,0.3906
7,data,students,0.3823
39,learn,machine,0.3446
16,models,machine,0.3406
10,data,researchers,0.3264
96,programming,computer,0.3223


### Q3(h): Distributional Hypothesis

The distributional hypothesis states that words occurring in similar contexts tend to have similar meanings or related usage.

The co-occurrence matrix represents each word by the words that occur around it. Two words with similar context distributions can therefore have similar vectors, which can be compared using cosine similarity.

This simple representation depends strongly on the corpus, context-window size, vocabulary and frequency of the words.
